# Sampling & Train/Test Data Manipulation

## Definition
Proper **sampling** and **train/test splitting** are critical to building valid ML models.
Pandas provides tools to sample data, shuffle, and create splits.

### Key Functions
| Function | Description |
|----------|-------------|
| `df.sample(n, frac)` | Random sampling |
| `df.sample(frac=1)` | Shuffle entire DataFrame |
| Boolean mask split | Deterministic index-based split |
| Stratified split | Preserve class distribution |
| Time-series split | Respect temporal ordering |

### Why This Matters
- **Random** splitting can cause data leakage in time-series
- **Stratified** splitting ensures class balance in train/test
- **Shuffling** before split avoids accidental ordering bias

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
n = 1000
df = pd.DataFrame({
    'feature_1': rng.standard_normal(n),
    'feature_2': rng.uniform(0, 10, n),
    'category': rng.choice(['A', 'B', 'C'], n, p=[0.5, 0.3, 0.2]),
    'target': rng.choice([0, 1], n, p=[0.7, 0.3])
})
print('Dataset:', df.shape)
print('Class distribution:', df['target'].value_counts(normalize=True).round(3).to_dict())

In [ ]:
# Simple random sampling
sample_100 = df.sample(n=100, random_state=42)
print('Random sample (n=100):', sample_100.shape)

sample_10pct = df.sample(frac=0.1, random_state=42)
print('10% sample:', sample_10pct.shape)

# Sampling with replacement (bootstrap)
bootstrap = df.sample(n=len(df), replace=True, random_state=42)
print('Bootstrap sample (with replace):', bootstrap.shape)

In [ ]:
# Shuffle entire DataFrame
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)
print('Shuffled (first 5 indices):', df_shuffled.index[:5].tolist())

In [ ]:
# Simple train/test split (80/20)
train_size = int(0.8 * len(df))
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)
train_df = df_shuffled[:train_size]
test_df  = df_shuffled[train_size:]
print(f'Train: {train_df.shape}, Test: {test_df.shape}')

In [ ]:
# Stratified split — preserve class distribution
from sklearn.model_selection import train_test_split

X = df[['feature_1', 'feature_2', 'category']]
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print('Train class dist:', y_train.value_counts(normalize=True).round(3).to_dict())
print('Test class dist: ', y_test.value_counts(normalize=True).round(3).to_dict())

In [ ]:
# Manual stratified split without sklearn
def stratified_split(df, target_col, test_frac=0.2, random_state=42):
    train_parts = []
    test_parts  = []
    for cls, group in df.groupby(target_col):
        shuffled = group.sample(frac=1, random_state=random_state)
        n_test = int(len(shuffled) * test_frac)
        test_parts.append(shuffled[:n_test])
        train_parts.append(shuffled[n_test:])
    train = pd.concat(train_parts).sample(frac=1, random_state=random_state).reset_index(drop=True)
    test  = pd.concat(test_parts).sample(frac=1, random_state=random_state).reset_index(drop=True)
    return train, test

train, test = stratified_split(df, 'target')
print(f'Train: {train.shape}, Test: {test.shape}')
print('Train dist:', train['target'].value_counts(normalize=True).round(3).to_dict())

In [ ]:
# Time-series split — NEVER shuffle time data!
ts_df = pd.DataFrame({
    'date': pd.date_range('2022-01-01', periods=365, freq='D'),
    'value': np.cumsum(np.random.randn(365))
})
ts_df = ts_df.sort_values('date')  # ensure time order

cutoff = pd.Timestamp('2022-10-01')
ts_train = ts_df[ts_df['date'] < cutoff]
ts_test  = ts_df[ts_df['date'] >= cutoff]
print(f'Time-series Train: {ts_train.shape} ({ts_train["date"].min().date()} to {ts_train["date"].max().date()})')
print(f'Time-series Test:  {ts_test.shape}  ({ts_test["date"].min().date()} to {ts_test["date"].max().date()})')